# Noite 1 · Ingestão bronze

A bronze preserva o dado como veio. **Nada de limpeza aqui**: se der problema
depois, a gente precisa poder voltar na origem e comparar.

## O poder do `read_files`

O **MESMO comando** lê CSV, planilha Excel, JSON e XML.
Ninguém converte nada antes. A Vértice tem projetos em três ferramentas:

| Ferramenta | Volume |
|------------|--------|
| **Excel** | 20 planilhas, uma por projeto, cada gerente do seu jeito |
| **ClickUp** | 25 exports da API em JSON (tarefas dentro de uma lista) |
| **MS Project** | 15 arquivos XML (uma `<Task>` por tarefa, com fases e baseline) |

## Duas coisas que valem explicar

- `inferColumnTypes => false` → **TUDO entra como texto** (CSV e Excel).
  Se o Databricks adivinhasse o tipo, ele erraria as datas que vêm em dois
  formatos e o valor com vírgula decimal. A sujeira sumiria antes de a
  gente ver. No JSON e no XML deixamos inferir: é o que mantém a estrutura
  (lista dentro de lista).
- `EXCEPT (_rescued_data)` → o `read_files` cria sozinho a coluna
  `_rescued_data`, a **"rede de segurança"**: o que não couber nas colunas
  (uma coluna a mais numa linha, por exemplo) cai ali em vez de sumir.
  Nos nossos arquivos ela vem vazia, então a gente tira da tabela.

## As 9 tabelas: uma pergunta do dono por trás de cada uma

| Tabelas | Pergunta |
|---------|----------|
| `projetos`, `status_semanal` | o cadastro do PMO e o farol de toda sexta |
| `excel_tarefas`, `clickup_tarefas`, `msproject_tarefas` | as 3 ferramentas (pergunta 1: quem atrasa) |
| `custos` | o financeiro (pergunta 2: quanto custa) |
| `colaboradores`, `apontamentos` | RH e timesheet (pergunta 3: gente parada) |
| `entregas_fornecedor` | compras (a consultoria que atrasa) |

## Antes de rodar

Os arquivos precisam estar no volume (passos 3 e 4 do README):

```
/Volumes/lakehouse_vertice/bronze/raw/<sistema>/...
```

Rode com:

```
python3 scripts/run_sql.py aulas/aula-01-analista/01-ingestao-bronze.sql --profile SEU-PERFIL
```

In [0]:
%sql
-- ---------- pmo ----------
-- projetos: o cadastro do PMO: cliente, valor, prazo e EM QUAL FERRAMENTA o projeto vive
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.projetos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/pmo/projetos.csv',
                format => 'csv', header => true, inferColumnTypes => false);
-- status_semanal: toda sexta o gerente publica avanço, farol e bloqueios (texto livre)
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.status_semanal AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/pmo/status_semanal.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- as três ferramentas de projeto ----------
-- excel_tarefas: as 20 planilhas de uma vez (é só apontar para a pasta).
-- A planilha tem cabeçalho escrito por gente ("% Concluído", "Início previsto"),
-- e tabela não aceita espaço no nome da coluna: renomeamos, sem mexer no valor.
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.excel_tarefas AS
SELECT `ID`               AS id,
       `Fase`             AS fase,
       `Tarefa`           AS tarefa,
       `Responsável`      AS responsavel,
       `Início previsto`  AS inicio_previsto,
       `Término previsto` AS termino_previsto,
       `Início real`      AS inicio_real,
       `Término real`     AS termino_real,
       `% Concluído`      AS pct_concluido,
       `Horas previstas`  AS horas_previstas,
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/excel/',
                format => 'excel', headerRows => 1, inferColumnTypes => false);

-- clickup_tarefas: cada arquivo é UMA lista (um projeto) com as tarefas dentro.
-- explode() abre a lista: uma linha por tarefa. O resto fica como veio:
-- status é uma estrutura, datas são milissegundos, campos personalizados são uma lista.
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.clickup_tarefas AS
SELECT list.id   AS list_id,
       list.name AS list_name,
       t.*,
       url_decode(_metadata.file_name) AS arquivo
FROM (
    SELECT list, explode(tasks) AS t, _metadata
    FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/clickup/',
                    format => 'json', multiLine => true)
);

-- msproject_tarefas: cada <Task> do XML vira uma linha. Inclui as tarefas-resumo
-- (as fases e o projeto inteiro), que somam as horas das tarefas de baixo.
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.msproject_tarefas AS
SELECT * EXCEPT (_rescued_data),
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/msproject/',
                format => 'xml', rowTag => 'Task');

-- ---------- rh e timesheet ----------
-- colaboradores: as 520 pessoas: papel, senioridade, custo/hora, admissão e desligamento
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.colaboradores AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/rh/colaboradores.csv',
                format => 'csv', header => true, inferColumnTypes => false);
-- apontamentos: horas disponíveis x horas apontadas, por pessoa, projeto e semana
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.apontamentos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/timesheet/apontamentos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- financeiro ----------
-- custos: cada lançamento de custo real (folha alocada, consultoria, cloud, licenças...)
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.custos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/financeiro/custos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- compras ----------
-- entregas_fornecedor: o que cada consultoria prometeu entregar, e quando entregou
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.entregas_fornecedor AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/compras/entregas_fornecedor.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- conferência ----------
-- 9 tabelas. Se alguma vier com 0 linhas, o arquivo não subiu para o volume.
SELECT 'projetos' AS tabela, COUNT(*) AS linhas FROM lakehouse_vertice.bronze.projetos
UNION ALL SELECT 'status_semanal', COUNT(*) FROM lakehouse_vertice.bronze.status_semanal
UNION ALL SELECT 'excel_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.excel_tarefas
UNION ALL SELECT 'clickup_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.clickup_tarefas
UNION ALL SELECT 'msproject_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.msproject_tarefas
UNION ALL SELECT 'colaboradores', COUNT(*) FROM lakehouse_vertice.bronze.colaboradores
UNION ALL SELECT 'apontamentos', COUNT(*) FROM lakehouse_vertice.bronze.apontamentos
UNION ALL SELECT 'custos', COUNT(*) FROM lakehouse_vertice.bronze.custos
UNION ALL SELECT 'entregas_fornecedor', COUNT(*) FROM lakehouse_vertice.bronze.entregas_fornecedor
ORDER BY linhas DESC;

In [0]:
%sql
-- ============================================
-- EXEMPLO 02: ORDER BY, CAST e DISTINCT — os maiores contratos
-- ============================================
-- Conceito: ORDER BY, CAST, DISTINCT
-- Pergunta de negócio: quais são os maiores contratos da Vértice?
-- Conexão: na bronze tudo é texto. Sem CAST, o Databricks ordena como palavra.

-- 1. Os 5 maiores contratos... ou não.
--    ORDER BY coluna DESC   ordena do maior para o menor
--    Deve aparecer: uma lista que PARECE certa, com o PRJ-052 (R$ 8,4 mi) no topo. Está errada.
--    O valor é texto, e texto se ordena letra por letra: '8380000' vem antes de '15275000'
--    porque '8' vem depois de '1', como numa lista telefônica.
SELECT projeto_id, nome_projeto, valor_contrato
FROM lakehouse_vertice.bronze.projetos
ORDER BY valor_contrato DESC
LIMIT 5;

-- 2. A correção: converter o texto em número antes de ordenar.
--    CAST(coluna AS DECIMAL(18,2))   transforma o texto em número com 2 casas decimais
--    Deve aparecer: o maior contrato de verdade é o PRJ-017, R$ 15,3 mi.
SELECT projeto_id, nome_projeto,
       CAST(valor_contrato AS DECIMAL(18,2)) AS valor_numero
FROM lakehouse_vertice.bronze.projetos
ORDER BY valor_numero DESC
LIMIT 5;

-- 3. Em quais ferramentas os projetos vivem?
--    DISTINCT   tira as repetições: cada valor aparece uma vez só
--    Deve aparecer: 8 jeitos de escrever 3 ferramentas: 'Excel', 'excel', 'Planilha',
--    'ClickUp', 'Clickup', 'MS Project', 'Project', 'msproject'.
--    É o retrato do problema da Vértice: cada um registra do seu jeito.
SELECT DISTINCT ferramenta
FROM lakehouse_vertice.bronze.projetos
ORDER BY ferramenta;

-- ============================================
-- EXEMPLO 03: WHERE — filtrar, e achar a sujeira
-- ============================================
-- Conceito: WHERE, LIKE, AND, IN, IS NULL
-- Pergunta de negócio: dá para confiar nesse número?
-- Conexão com a noite 2: tudo que este exemplo acha é o que a silver conserta.
--
-- Dado sujo não dá erro: dá um número errado, em silêncio, e a diretoria decide em cima dele.
--
-- O WHERE vem depois do FROM e diz QUAIS linhas entram:
--   WHERE coluna = 'valor'      igual a
--   WHERE coluna LIKE '%/%'     "parece com". O % quer dizer "qualquer coisa aqui":
--                               '%/%' = tem uma barra em algum lugar

-- 1. A data de início em dois formatos na mesma coluna: 14/03/2025 e 2025-03-14.
--    Quantos projetos estão no formato com barra?
--    Deve aparecer: 24 (os outros 36 estão como 2025-03-14). Uma conta de prazo que só
--    entenda um dos formatos erra 24 projetos.
SELECT COUNT(*) AS formato_brasileiro
FROM lakehouse_vertice.bronze.projetos
WHERE data_inicio LIKE '%/%';

-- 2. Excel: a linha de TOTAL que o gerente deixou no fim de cada planilha.
--    Deve aparecer: 20 linhas, uma por planilha. Quem soma a coluna de horas inteira
--    conta tudo duas vezes.
SELECT arquivo, tarefa, horas_previstas
FROM lakehouse_vertice.bronze.excel_tarefas
WHERE tarefa = 'TOTAL';

-- 3. Financeiro: custo com vírgula decimal ('12345,67'). Para o computador, isso não é número.
--    Deve aparecer: 671 lançamentos. O exemplo 04 mostra o estrago.
SELECT COUNT(*) AS com_virgula
FROM lakehouse_vertice.bronze.custos
WHERE valor LIKE '%,%';

-- 4. Combinar condições: projetos de VAREJO que estão EM ANDAMENTO.
--    AND   as duas condições precisam valer ao mesmo tempo
--    Deve aparecer: 14 projetos. Varejo é o segmento da Black Friday: o freeze de 01/11 a 10/01 trava os de e-commerce.
SELECT projeto_id, cliente, nome_projeto
FROM lakehouse_vertice.bronze.projetos
WHERE status = 'em_andamento'
  AND segmento = 'varejo';

-- 5. Os projetos que vivem no Excel. Lembra que a ferramenta está escrita de 8 jeitos (exemplo 02)?
--    IN (...)   "é um destes valores": vale como três igualdades ligadas por OR
--    Deve aparecer: 20 projetos. Com WHERE ferramenta = 'Excel' sairiam menos: os escritos
--    'excel' e 'Planilha' ficariam de fora, sem nenhum aviso.
SELECT projeto_id, cliente, ferramenta
FROM lakehouse_vertice.bronze.projetos
WHERE ferramenta IN ('Excel', 'excel', 'Planilha');

-- 6. Projeto sem o código da ferramenta: ninguém sabe onde está o cronograma dele.
--    IS NULL   a coluna está vazia. Atenção: "= NULL" não funciona, nunca dá verdadeiro.
--    Deve aparecer: 2 projetos (PRJ-009 e PRJ-012).
SELECT projeto_id, nome_projeto, ferramenta, codigo_ferramenta
FROM lakehouse_vertice.bronze.projetos
WHERE codigo_ferramenta IS NULL;

-- 7. A mesma consultoria digitada de vários jeitos em compras.
--    lower(...)   põe tudo em minúscula, para o LIKE achar 'Forge', 'FORGE' e 'forge'
--    Deve aparecer: 4 jeitos de escrever CodeForge. Quem soma "por fornecedor" acha 4 fornecedores.
SELECT DISTINCT fornecedor
FROM lakehouse_vertice.bronze.entregas_fornecedor
WHERE lower(fornecedor) LIKE '%forge%';

-- ============================================
-- EXEMPLO 04: GROUP BY — para onde vai o dinheiro da Vértice?
-- ============================================
-- Conceito: GROUP BY, SUM, replace
-- Pergunta de negócio: quanto a Vértice gasta, por categoria de custo?
-- Conexão: é a pergunta 2 do dono ("quanto custa?") começando a ter resposta.
--
-- A tabela é a bronze.custos: um lançamento de custo por linha (23.640 linhas).

-- 1. Quanto cada categoria gastou?
--    GROUP BY categoria   junta as linhas da mesma categoria e faz a conta dentro de cada grupo
--    SUM(...)             soma a coluna dentro do grupo
--    O valor é texto, então primeiro CAST para número (exemplo 02).
--    Deve aparecer: um ERRO. É de propósito: '1827,44' não vira número. É a vírgula do exemplo 03.
SELECT categoria,
       SUM(CAST(valor AS DECIMAL(18,2))) AS custo
FROM lakehouse_vertice.bronze.custos
GROUP BY categoria;

-- 2. A correção: trocar a vírgula por ponto ANTES de converter.
--    replace(valor, ',', '.')   troca ',' por '.'
--    ROUND(x / 1000000, 1)      em milhões, com 1 casa decimal
--    Deve aparecer: pessoal no topo, com R$ 117,8 mi. Empresa de tecnologia gasta com gente:
--    quase dois terços dos R$ 193,35 mi de custo.
SELECT categoria,
       ROUND(SUM(CAST(replace(valor, ',', '.') AS DECIMAL(18,2))) / 1000000, 1) AS custo_milhoes
FROM lakehouse_vertice.bronze.custos
GROUP BY categoria
ORDER BY custo_milhoes DESC;

-- ============================================
-- EXEMPLO 05: UNION — as três ferramentas numa fila só
-- ============================================
-- Conceito: UNION ALL, VIEW, CASE, JOIN, WITH
-- Pergunta de negócio: QUAIS PROJETOS ESTÃO MAIS ATRASADOS?
--
-- É a primeira pergunta do dono, e ninguém na Vértice sabia responder: 20 projetos estão
-- no Excel, 25 no ClickUp e 15 no MS Project. Cada ferramenta guarda a mesma informação
-- (a tarefa, quando devia começar e terminar, quanto está pronta, quantas horas) de um jeito.
--
-- O plano, em 5 degraus:
--   bloco 1   o UNION ALL no menor exemplo possível
--   bloco 2   traduzir UMA ferramenta (o Excel) para as 7 colunas
--   bloco 3   as três traduzidas e empilhadas numa view: o rascunho
--   bloco 4   somar as horas: quanto está pronto e quanto devia estar
--   bloco 5   dividir uma pela outra (o SPI) e dar a resposta ao dono
-- É um RASCUNHO feito à mão. Na noite 2 isso vira pipeline de verdade (a silver).
--
-- O "hoje" do dataset é 16/10/2026: nunca use current_date() aqui,
-- senão o seu número muda todo dia e não bate com o de ninguém.

-- 1. O UNION ALL, no menor exemplo possível.
--    UNION ALL   empilha o resultado de uma consulta embaixo da outra.
--                Regra: todas com as mesmas colunas, na mesma ordem.
--    Deve aparecer: 3 linhas, uma por ferramenta, com quantas linhas cada uma tem.
--    É essa a ideia do exemplo inteiro. O resto é fazer as três ferramentas
--    terem as mesmas colunas, para poder empilhar.
SELECT 'excel'     AS ferramenta, COUNT(*) AS linhas FROM lakehouse_vertice.bronze.excel_tarefas
UNION ALL
SELECT 'clickup'   AS ferramenta, COUNT(*) AS linhas FROM lakehouse_vertice.bronze.clickup_tarefas
UNION ALL
SELECT 'msproject' AS ferramenta, COUNT(*) AS linhas FROM lakehouse_vertice.bronze.msproject_tarefas;

-- 2. Traduzir UMA ferramenta: o Excel nas 7 colunas.
--
--    O QUE A GENTE QUER: que cada linha da planilha vire uma tarefa "limpa", com as mesmas
--    7 colunas que o ClickUp e o MS Project vão ter. Só assim dá para empilhar (bloco 1).
--      projeto_id · ferramenta · tarefa · inicio_baseline · fim_baseline · pct_concluido · horas
--      (baseline = o plano original: quando a tarefa DEVIA começar e terminar)
--
--    O QUE ATRAPALHA (você viu tudo isso nos exemplos 01 e 03):
--      · a planilha não tem coluna de projeto: o PRJ-xxx está no nome do arquivo
--      · a data vem de dois jeitos: '06/03/2025' ou 45700 (número de série do Excel,
--        que conta os dias desde 30/12/1899)
--      · o % vem de dois jeitos: '45%' (às vezes '64,4%', com vírgula) ou 0.45; e tem % acima de 100
--      · as horas vêm como texto, às vezes com 'h' ('64h')
--      · a última linha de cada planilha é o TOTAL
--
--    O COMANDO NOVO: CASE WHEN condição THEN isto ELSE aquilo END, o "se" do SQL.
--      Ele resolve os dois jeitos: SE tem barra, lê como data; SENÃO, é número de série.
--
--    TENTE COM A IA: no SQL Editor, abra o Assistant (o ícone de estrelinha) e cole o prompt:
--
--      Escreva um SELECT na tabela lakehouse_vertice.bronze.excel_tarefas, com LIMIT 10,
--      que devolva 7 colunas: projeto_id, ferramenta, tarefa, inicio_baseline, fim_baseline,
--      pct_concluido e horas. Regras:
--      - projeto_id: extraia o padrão PRJ- seguido de 3 dígitos da coluna arquivo
--      - ferramenta: o texto fixo 'excel'
--      - inicio_baseline e fim_baseline vêm de inicio_previsto e termino_previsto, que são texto.
--        Se tiver barra, é uma data dd/MM/yyyy. Senão, é o número de série do Excel: dias
--        contados a partir de 1899-12-30
--      - pct_concluido: se tiver o símbolo %, tire o % e troque vírgula por ponto. Senão, é
--        uma fração (0.45): multiplique por 100. Limite o resultado a no máximo 100
--      - horas: horas_previstas sem a letra h, como número
--      - ignore as linhas em que tarefa = 'TOTAL'
--      Use CASE WHEN e escreva um comentário curto em português antes de cada coluna.
--
--    Compare o que a IA escreveu com o gabarito abaixo. Rode os dois: o resultado bate?
--    Deve aparecer: datas de verdade (mesmo onde a planilha tinha 45700) e % de 0 a 100.
SELECT -- o projeto está no nome do arquivo: 'Cronograma PRJ-002.xlsx' → pega "PRJ-" + 3 dígitos
       regexp_extract(arquivo, 'PRJ-[0-9]{3}', 0)                         AS projeto_id,
       'excel'                                                            AS ferramenta,
       tarefa,
       -- data com barra ('06/03/2025') → lê como dd/MM/yyyy;
       -- senão é o número de série do Excel (45700 = dias contados desde 30/12/1899)
       CASE WHEN inicio_previsto LIKE '%/%' THEN to_date(inicio_previsto, 'dd/MM/yyyy')
            ELSE date_add(DATE'1899-12-30', CAST(inicio_previsto AS INT)) END   AS inicio_baseline,
       CASE WHEN termino_previsto LIKE '%/%' THEN to_date(termino_previsto, 'dd/MM/yyyy')
            ELSE date_add(DATE'1899-12-30', CAST(termino_previsto AS INT)) END  AS fim_baseline,
       -- '45%' ou '64,4%' → 45 e 64.4;  0.45 → 45;  e LEAST corta quem passou de 100
       LEAST(100, CASE WHEN pct_concluido LIKE '%\%%'
                       THEN CAST(replace(replace(pct_concluido, '%', ''), ',', '.') AS DOUBLE)
                       ELSE CAST(pct_concluido AS DOUBLE) * 100 END)      AS pct_concluido,
       CAST(replace(horas_previstas, 'h', '') AS DOUBLE)                  AS horas
FROM lakehouse_vertice.bronze.excel_tarefas
WHERE tarefa <> 'TOTAL'                     -- fora a linha de TOTAL (exemplo 03)
LIMIT 10;

-- 3. O rascunho: as três ferramentas traduzidas e empilhadas, guardadas numa view.
--    CREATE OR REPLACE VIEW nome AS ...   guarda a consulta com um nome. Não copia o dado:
--                                         toda vez que alguém lê a view, a consulta roda de novo.
--    O Excel é o bloco 2. O ClickUp e o MS Project fazem a mesma tradução, cada um com o seu
--    problema (explicado no comentário de cada parte). Entre eles, o UNION ALL do bloco 1.
--    Deve aparecer: OK. A view bronze.rascunho_tarefas aparece no Catalog.
CREATE OR REPLACE VIEW lakehouse_vertice.bronze.rascunho_tarefas
COMMENT 'Rascunho da noite 1: tarefas do Excel, ClickUp e MS Project empilhadas à mão. Na noite 2 vira silver.tarefas.'
AS
-- ---------- EXCEL ----------
SELECT -- o projeto está no nome do arquivo: 'Cronograma PRJ-002.xlsx' → pega "PRJ-" + 3 dígitos
       regexp_extract(arquivo, 'PRJ-[0-9]{3}', 0)                         AS projeto_id,
       'excel'                                                            AS ferramenta,
       tarefa,
       -- data com barra ('06/03/2025') → lê como dd/MM/yyyy;
       -- senão é o número de série do Excel (45700 = dias contados desde 30/12/1899)
       CASE WHEN inicio_previsto LIKE '%/%' THEN to_date(inicio_previsto, 'dd/MM/yyyy')
            ELSE date_add(DATE'1899-12-30', CAST(inicio_previsto AS INT)) END   AS inicio_baseline,
       CASE WHEN termino_previsto LIKE '%/%' THEN to_date(termino_previsto, 'dd/MM/yyyy')
            ELSE date_add(DATE'1899-12-30', CAST(termino_previsto AS INT)) END  AS fim_baseline,
       -- '45%' ou '64,4%' → 45 e 64.4;  0.45 → 45;  e LEAST corta quem passou de 100
       LEAST(100, CASE WHEN pct_concluido LIKE '%\%%'
                       THEN CAST(replace(replace(pct_concluido, '%', ''), ',', '.') AS DOUBLE)
                       ELSE CAST(pct_concluido AS DOUBLE) * 100 END)      AS pct_concluido,
       CAST(replace(horas_previstas, 'h', '') AS DOUBLE)                  AS horas
FROM lakehouse_vertice.bronze.excel_tarefas
WHERE tarefa <> 'TOTAL'                     -- fora a linha de TOTAL (exemplo 03)

UNION ALL
-- ---------- CLICKUP ----------
-- O ClickUp não sabe o que é PRJ-xxx. O cadastro do PMO sabe: guarda o código da lista do ClickUp.
-- O JOIN junta as duas tabelas (c = clickup, p = projetos); o OR casa pelo nome quando falta o código.
-- O % e as datas da baseline estão escondidos numa lista de campos personalizados: o filter()
-- procura o campo pelo nome. As datas e as horas vêm em milissegundos.
SELECT p.projeto_id,
       'clickup',
       c.name,
       to_date(timestamp_millis(CAST(filter(c.custom_fields, f -> f.name = 'Início baseline')[0].value AS BIGINT))),
       to_date(timestamp_millis(CAST(filter(c.custom_fields, f -> f.name = 'Fim baseline')[0].value AS BIGINT))),
       CAST(filter(c.custom_fields, f -> f.name = 'Progresso (%)')[0].value AS DOUBLE),
       c.time_estimate / 3600000                                           -- milissegundos → horas
FROM lakehouse_vertice.bronze.clickup_tarefas c
JOIN lakehouse_vertice.bronze.projetos p
  ON p.codigo_ferramenta = c.list_id OR p.nome_projeto = c.list_name

UNION ALL
-- ---------- MS PROJECT ----------
-- As datas da baseline estão dentro da "caixinha" Baseline; as horas vêm como 'PT40H0M0S' (→ 40).
-- Summary = 0: só as tarefas de verdade. A tarefa-resumo (a fase) soma as de baixo e contaria duas vezes.
SELECT regexp_extract(arquivo, 'PRJ-[0-9]{3}', 0),
       'msproject',
       Name,
       to_date(Baseline.Start),
       to_date(Baseline.Finish),
       CAST(PercentComplete AS DOUBLE),
       CAST(regexp_extract(Work, 'PT([0-9]+)H', 1) AS DOUBLE)
FROM lakehouse_vertice.bronze.msproject_tarefas
WHERE Summary = 0;

-- 4. Com tudo numa fila só, as horas de cada projeto.
--    horas agregadas  = quanto ESTÁ pronto:        horas da tarefa × % concluído
--    horas planejadas = quanto DEVIA estar pronto:  horas da tarefa × quanto do prazo da tarefa
--                                                   já passou até hoje (de 0 a 1)
--    datediff(fim, inicio)      dias entre duas datas
--    GREATEST(0, LEAST(1, x))   prende x entre 0 e 1: tarefa que nem começou = 0; que já devia ter acabado = 1
--    Deve aparecer: dois números por projeto. Onde o primeiro é bem menor que o segundo, atrasou.
SELECT projeto_id,
       ferramenta,
       ROUND(SUM(horas * pct_concluido / 100)) AS horas_agregadas,
       ROUND(SUM(horas * GREATEST(0, LEAST(1,
           datediff(DATE'2026-10-16', inicio_baseline) / datediff(fim_baseline, inicio_baseline)
       ))))                                     AS horas_planejadas
FROM lakehouse_vertice.bronze.rascunho_tarefas
GROUP BY projeto_id, ferramenta
ORDER BY projeto_id;

-- 5. A resposta para o dono: o SPI de cada projeto em andamento.
--    SPI = agregadas ÷ planejadas      abaixo de 1 é atraso (0,80 = andou 80% do que devia)
--    Regra da Torre: abaixo de 0,9 = atenção; abaixo de 0,8 = crítico.
--
--    WITH valor AS (...)          dá um nome à consulta do bloco 4 e usa embaixo, como se fosse
--                                 uma tabela: primeiro soma, depois divide
--    JOIN ... USING (projeto_id)  junta com o cadastro, para trazer o cliente e o status
--    Deve aparecer: 41 projetos em andamento. No topo, PRJ-016 (0,63), PRJ-030 (0,69),
--    PRJ-039 (0,75) e PRJ-011 (0,77). A resposta da pergunta 1.
WITH valor AS (
    SELECT projeto_id,
           ferramenta,
           SUM(horas * pct_concluido / 100) AS horas_agregadas,
           SUM(horas * GREATEST(0, LEAST(1,
               datediff(DATE'2026-10-16', inicio_baseline) / datediff(fim_baseline, inicio_baseline)
           )))                               AS horas_planejadas
    FROM lakehouse_vertice.bronze.rascunho_tarefas
    GROUP BY projeto_id, ferramenta
)
SELECT p.projeto_id,
       p.cliente,
       v.ferramenta,
       ROUND(v.horas_agregadas / v.horas_planejadas, 2) AS spi,
       CASE WHEN v.horas_agregadas / v.horas_planejadas < 0.8 THEN 'crítico'
            WHEN v.horas_agregadas / v.horas_planejadas < 0.9 THEN 'atenção'
            ELSE 'ok' END                               AS situacao_prazo
FROM lakehouse_vertice.bronze.projetos p
JOIN valor v USING (projeto_id)
WHERE p.status = 'em_andamento'
ORDER BY spi;